# 02B — Pemodelan LSTM pada Data Baru (v2): 3 Skenario Simulasi Ketimpangan

Notebook ini menguji performa model LSTM unidirectional pada 3 skenario rasio ketimpangan data latih:
1. **Skenario 1:1:1** (Seimbang Sempurna: 1.000 Neg, 1.000 Net, 1.000 Pos, Total 3.000 sampel)
2. **Skenario 6:3:1** (Ketimpangan Moderat: 3.000 Neg, 500 Net, 1.500 Pos, Total 5.000 sampel)
3. **Skenario 8:1:1** (Ketimpangan Ekstrem / Long-tail: 3.200 Neg, 400 Net, 400 Pos, Total 4.000 sampel)

Seluruh model dievaluasi secara adil (*apple-to-apple*) pada **Data Uji Empiris Terkunci ($n = 1.730$)** dari `banjir_processed_v2.csv`.


In [ ]:
import os
import sys
from pathlib import Path

def resolve_path(filename):
    """Cari file di /kaggle/input (Kaggle) atau fallback ke path lokal."""
    # 1. Kaggle environment: walk /kaggle/input
    if Path("/kaggle/input").exists():
        for root, _dirs, files in os.walk("/kaggle/input"):
            if filename in files:
                found = os.path.join(root, filename)
                print(f"[resolve_path] Ditemukan di Kaggle: {found}")
                return found
    # 2. Fallback lokal
    candidates = [
        Path(f"Data/processed/{filename}"),
        Path(f"Data/simulated/{filename}"),
        Path(f"Data/{filename}"),
        Path(f"Output/predictions/{filename}"),
        Path(f"../Data/processed/{filename}"),
        Path(f"../Data/simulated/{filename}"),
        Path(f"../Data/{filename}"),
        Path(filename),
    ]
    for p in candidates:
        if p.exists():
            print(f"[resolve_path] Ditemukan lokal: {p}")
            return str(p)
    return filename

print('Fungsi resolve_path siap.')


In [ ]:
import numpy as np
import pandas as pd
import random
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dropout, Dense
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, recall_score, precision_score

# Kunci seed deterministik
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Load Test Set Empiris Terkunci (20% Stratified Split, Seed 42)
csv_path = resolve_path('banjir_processed_v2.csv')
df_full = pd.read_csv(csv_path)
col_text = 'processed_text_v2'
col_label = 'label'

_, test_df = train_test_split(df_full, test_size=0.20, stratify=df_full[col_label], random_state=SEED)
X_test_raw = test_df[col_text].astype(str).values
y_test = test_df[col_label].values

print(f'Test set terkunci: {len(test_df)} sampel')
print('Distribusi test label:', pd.Series(y_test).value_counts().sort_index().to_dict())


In [ ]:
def build_lstm_model(vocab_size=10000, embedding_dim=128, units=64, dropout=0.2, max_len=50, lr=0.0002):
    model = Sequential([
        Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len),
        LSTM(units),
        Dropout(dropout),
        Dense(3, activation='softmax')
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

print('Arsitektur LSTM siap.')


In [ ]:
# Eksekusi Pelatihan & Evaluasi pada 3 Skenario Simulasi
scenarios = ['111', '631', '811']
results = []
all_predictions = {}

for sc in scenarios:
    sc_file = f'scenario_{sc}.csv'
    sc_path = resolve_path(sc_file)
    print('=' * 65)
    print(f'MEMPROSES SKENARIO {sc} DARI: {sc_path}')
    print('=' * 65)
    
    df_sc = pd.read_csv(sc_path)
    print(f'Jumlah data: {len(df_sc)} baris | Distribusi: {df_sc[col_label].value_counts().to_dict()}')
    
    # Split 90% train, 10% validation stratified
    tr_sub, val_sub = train_test_split(df_sc, test_size=0.10, stratify=df_sc[col_label], random_state=SEED)
    
    tokenizer = Tokenizer(num_words=10000, oov_token='<OOV>')
    tokenizer.fit_on_texts(tr_sub[col_text].astype(str))
    
    X_train_seq = pad_sequences(tokenizer.texts_to_sequences(tr_sub[col_text].astype(str)), maxlen=50, padding='post', truncating='post')
    X_val_seq = pad_sequences(tokenizer.texts_to_sequences(val_sub[col_text].astype(str)), maxlen=50, padding='post', truncating='post')
    X_test_seq = pad_sequences(tokenizer.texts_to_sequences(X_test_raw), maxlen=50, padding='post', truncating='post')
    
    y_train = tr_sub[col_label].values
    y_val = val_sub[col_label].values
    
    tf.keras.backend.clear_session()
    tf.random.set_seed(SEED)
    np.random.seed(SEED)
    
    model = build_lstm_model()
    early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
    
    history = model.fit(
        X_train_seq, y_train,
        validation_data=(X_val_seq, y_val),
        epochs=20,
        batch_size=16,
        callbacks=[early_stop],
        verbose=1
    )
    
    # Prediksi Test Set
    probs = model.predict(X_test_seq, verbose=0)
    y_pred = np.argmax(probs, axis=1)
    all_predictions[sc] = y_pred
    
    acc = accuracy_score(y_test, y_pred)
    f1_mac = f1_score(y_test, y_pred, average='macro', zero_division=0)
    rec_mac = recall_score(y_test, y_pred, average='macro', zero_division=0)
    rec_per_cls = recall_score(y_test, y_pred, average=None, zero_division=0)
    rec_netral = rec_per_cls[1] if len(rec_per_cls) > 1 else 0.0
    
    print(f'\n[Hasil Test Skenario {sc}]')
    print(f'Accuracy      : {acc*100:.2f}%')
    print(f'Macro F1      : {f1_mac*100:.2f}%')
    print(f'Recall Netral : {rec_netral*100:.2f}%')
    print('Classification Report:')
    print(classification_report(y_test, y_pred, target_names=['negatif', 'netral', 'positif'], zero_division=0))
    
    results.append({
        'Skenario': sc,
        'Accuracy': acc,
        'Macro_F1': f1_mac,
        'Macro_Recall': rec_mac,
        'Recall_Netral': rec_netral
    })


In [ ]:
# Rangkuman & Ekspor Hasil Simulasi LSTM
df_results = pd.DataFrame(results)
df_results['Accuracy (%)'] = (df_results['Accuracy'] * 100).round(2)
df_results['Macro F1 (%)'] = (df_results['Macro_F1'] * 100).round(2)
df_results['Recall Netral (%)'] = (df_results['Recall_Netral'] * 100).round(2)

print('=' * 65)
print('TABEL RANGKUMAN HASIL SIMULASI LSTM (DATA BARU V2)')
print('=' * 65)
print(df_results[['Skenario', 'Accuracy (%)', 'Macro F1 (%)', 'Recall Netral (%)']].to_string(index=False))

out_dir = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('Output/simulated')
out_dir.mkdir(parents=True, exist_ok=True)
df_results.to_csv(out_dir / 'results_lstm_simulasi.csv', index=False)
print(f'Hasil tersimpan di {out_dir / "results_lstm_simulasi.csv"}')


In [ ]:
# Visualisasi Confusion Matrix untuk 3 Skenario
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
label_names = ['Negatif', 'Netral', 'Positif']

for idx, sc in enumerate(scenarios):
    cm = confusion_matrix(y_test, all_predictions[sc])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx], xticklabels=label_names, yticklabels=label_names)
    axes[idx].set_title(f'Skenario {sc}')
    axes[idx].set_xlabel('Prediksi')
    axes[idx].set_ylabel('Aktual')

plt.tight_layout()
plt.savefig(out_dir / 'confusion_matrix_lstm_simulasi.png', dpi=300)
plt.show()
